# 05 Final Load Prep

Use this notebook to compute final KPIs, prepare the Tableau-ready dataset, and export the exact file used in the dashboard.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [ ]:
DELIVERED   = PROJECT_ROOT / 'data/processed/olist_delivered_orders_master.csv'
FULL_MASTER = PROJECT_ROOT / 'data/processed/olist_orders_master.csv'
KPI_DIR     = PROJECT_ROOT / 'data/processed'

df         = pd.read_csv(DELIVERED,   parse_dates=['order_purchase_timestamp'])
orders_all = pd.read_csv(FULL_MASTER, parse_dates=['order_purchase_timestamp'])

print(f'Delivered orders : {df.shape[0]:,}')
print(f'All orders       : {orders_all.shape[0]:,}')
df.head(3)

In [ ]:
monthly_kpis = (
    df.groupby('order_purchase_month')
    .agg(
        total_revenue      =('order_value',      'sum'),
        order_count        =('order_id',         'count'),
        avg_order_value    =('order_value',      'mean'),
        late_delivery_rate =('is_late_delivery', 'mean'),
        avg_review_score   =('review_score',     'mean'),
        avg_delivery_days  =('delivery_days',    'mean'),
    )
    .round(4)
    .reset_index()
    .rename(columns={'order_purchase_month': 'month'})
)

print(f'Monthly KPIs computed across {len(monthly_kpis)} months')
monthly_kpis.tail(6)

In [ ]:
category_kpis = (
    df.groupby('primary_product_category')
    .agg(
        total_revenue      =('order_value',      'sum'),
        order_count        =('order_id',         'count'),
        avg_order_value    =('order_value',      'mean'),
        avg_review_score   =('review_score',     'mean'),
        late_delivery_rate =('is_late_delivery', 'mean'),
        avg_delivery_days  =('delivery_days',    'mean'),
    )
    .round(4)
    .sort_values('total_revenue', ascending=False)
    .reset_index()
    .rename(columns={'primary_product_category': 'product_category'})
)

print(f'Category KPIs: {len(category_kpis)} unique categories')
category_kpis.head(10)

In [ ]:
state_kpis = (
    df.groupby('customer_state')
    .agg(
        order_count        =('order_id',         'count'),
        total_revenue      =('order_value',      'sum'),
        avg_order_value    =('order_value',      'mean'),
        avg_review_score   =('review_score',     'mean'),
        late_delivery_rate =('is_late_delivery', 'mean'),
        avg_delivery_days  =('delivery_days',    'mean'),
    )
    .round(4)
    .sort_values('order_count', ascending=False)
    .reset_index()
    .rename(columns={'customer_state': 'state'})
)

print(f'State KPIs: {len(state_kpis)} states')
state_kpis.head(8)

In [ ]:
payment_kpis = (
    df.groupby('payment_type_primary')
    .agg(
        order_count      =('order_id',                 'count'),
        total_revenue    =('order_value',              'sum'),
        avg_order_value  =('order_value',              'mean'),
        avg_installments =('payment_installments_max', 'mean'),
    )
    .round(4)
    .reset_index()
    .rename(columns={'payment_type_primary': 'payment_type'})
)

total_orders  = payment_kpis['order_count'].sum()
total_revenue = payment_kpis['total_revenue'].sum()
payment_kpis['order_share_pct']   = (payment_kpis['order_count']   / total_orders  * 100).round(2)
payment_kpis['revenue_share_pct'] = (payment_kpis['total_revenue'] / total_revenue * 100).round(2)

payment_kpis.sort_values('order_count', ascending=False)

In [ ]:
seller_state_kpis = (
    df.groupby('primary_seller_state')
    .agg(
        order_count        =('order_id',         'count'),
        avg_delivery_days  =('delivery_days',    'mean'),
        avg_carrier_lag    =('carrier_lag_days', 'mean'),
        late_delivery_rate =('is_late_delivery', 'mean'),
        avg_review_score   =('review_score',     'mean'),
    )
    .round(4)
    .query('order_count >= 100')
    .sort_values('avg_delivery_days')
    .reset_index()
    .rename(columns={'primary_seller_state': 'seller_state'})
)

print(f'Seller state KPIs: {len(seller_state_kpis)} states (>=100 orders)')
seller_state_kpis.head(8)

In [ ]:
monthly_ref = monthly_kpis[
    ['month', 'avg_order_value', 'late_delivery_rate',
     'avg_review_score', 'avg_delivery_days']
].rename(columns={
    'avg_order_value'   : 'month_avg_order_value',
    'late_delivery_rate': 'month_late_delivery_rate',
    'avg_review_score'  : 'month_avg_review_score',
    'avg_delivery_days' : 'month_avg_delivery_days',
})

tableau_df = orders_all.merge(
    monthly_ref,
    left_on='order_purchase_month',
    right_on='month',
    how='left'
).drop(columns=['month'])

drop_cols = [c for c in tableau_df.columns
             if c in ('review_comment_title', 'review_comment_message')]
tableau_df = tableau_df.drop(columns=drop_cols)

print(f'Tableau-ready dataset: {tableau_df.shape[0]:,} rows x {tableau_df.shape[1]} columns')
tableau_df.head(3)

In [ ]:
KPI_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    'kpi_monthly.csv'          : monthly_kpis,
    'kpi_by_category.csv'      : category_kpis,
    'kpi_by_state.csv'         : state_kpis,
    'kpi_by_payment_type.csv'  : payment_kpis,
    'kpi_by_seller_state.csv'  : seller_state_kpis,
    'tableau_ready_dataset.csv': tableau_df,
}

for filename, frame in exports.items():
    frame.to_csv(KPI_DIR / filename, index=False)
    print(f'  {filename:<35} -> {len(frame):>6} rows x {len(frame.columns)} cols')

print('\nDone.')

## Final Load — Summary

### KPI Tables Written to `data/processed/`

| File | Granularity | Use In Tableau |
|---|---|---|
| `kpi_monthly.csv` | One row per calendar month | Revenue trend, late delivery trend line |
| `kpi_by_category.csv` | One row per product category | Category revenue bar chart, satisfaction heatmap |
| `kpi_by_state.csv` | One row per customer state | Geographic map view, state ranking table |
| `kpi_by_payment_type.csv` | One row per payment method | Payment mix pie / bar chart |
| `kpi_by_seller_state.csv` | One row per seller state (≥100 orders) | Seller-side delivery performance map |
| `tableau_ready_dataset.csv` | One row per order (all statuses) | Main Tableau data source — use extracts |

### Headline KPIs (delivered orders baseline)

Run the cells above to see the exact computed values. Expected ranges based on the dataset:

- **Total GMV (delivered orders)**: ~R$13–15 million
- **Average order value**: ~R$154–160
- **Median delivery time**: ~12 days
- **Late delivery rate**: ~8.1%
- **Average review score**: ~4.13 / 5
- **Credit card order share**: ~74%